# Generation Step Iterator Recipe

Large language-model `generate()` methods hide a loop over decoding steps. The practical TorchLens pattern is to make that loop explicit, decide which hook plan applies at each step, and capture one step at a time. This notebook uses a tiny local generator with a `generate` method so the pattern is visible without downloading a model.


In [ ]:
from __future__ import annotations

from collections.abc import Callable, Mapping

import torch
from torch import nn

import torchlens as tl

HookPlan = Mapping[object, object] | None


class TinyTokenGenerator(nn.Module):
    """Minimal token generator with a visible decode loop."""

    def __init__(self) -> None:
        """Initialize deterministic token model weights."""

        super().__init__()
        self.embed = nn.Embedding(5, 4)
        self.transition = nn.Linear(4, 4)
        self.head = nn.Linear(4, 5)
        with torch.no_grad():
            self.embed.weight.zero_()
            self.transition.weight.zero_()
            self.transition.bias.zero_()
            self.head.weight.zero_()
            self.head.bias.zero_()
            self.head.bias[1] = 0.1
            self.head.weight[3, 0] = 1.0

    def forward(self, token: torch.Tensor) -> torch.Tensor:
        """Score the next token from the current token.

        Parameters
        ----------
        token:
            Current token IDs with shape ``(batch,)``.

        Returns
        -------
        torch.Tensor
            Next-token logits with shape ``(batch, vocab)``.
        """

        hidden = torch.relu(self.transition(self.embed(token)))
        return self.head(hidden)

    def generate(
        self,
        start_token: torch.Tensor,
        *,
        steps: int,
        hook_factory: Callable[[int], HookPlan] | None = None,
    ) -> tuple[torch.Tensor, list[tl.ModelLog]]:
        """Generate tokens while optionally applying per-step TorchLens hooks.

        Parameters
        ----------
        start_token:
            Initial token IDs.
        steps:
            Number of decode steps.
        hook_factory:
            Optional callable returning a hook mapping for each step.

        Returns
        -------
        tuple[torch.Tensor, list[tl.ModelLog]]
            Generated token sequence and one TorchLens log per step.
        """

        current = start_token.clone()
        tokens = [current]
        logs: list[tl.ModelLog] = []
        for step in range(steps):
            hooks = None if hook_factory is None else hook_factory(step)
            step_log = tl.log_forward_pass(
                self,
                current,
                vis_opt="none",
                intervention_ready=True,
                hooks=hooks,
            )
            logits = step_log.layer_list[-1].activation.detach()
            current = logits.argmax(dim=-1)
            tokens.append(current)
            logs.append(step_log)
        return torch.stack(tokens, dim=1), logs


model = TinyTokenGenerator().eval()
start = torch.tensor([0])

The clean decode picks token `1` at every step because the output bias favors it. The steered decode adds a positive feature at the ReLU site, which makes token `3` win. This mirrors a per-token hook policy around `model.generate()`.


In [ ]:
clean_tokens, clean_logs = model.generate(start, steps=3)
print("clean", clean_tokens)

steering_direction = torch.tensor([1.0, 0.0, 0.0, 0.0])


def hooks_for_step(step: int) -> HookPlan:
    """Return the hook plan for one decode step.

    Parameters
    ----------
    step:
        Zero-based generation step.

    Returns
    -------
    HookPlan
        Mapping from TorchLens site selector to helper.
    """

    magnitude = 1.0 if step >= 0 else 0.0
    return {tl.func("relu"): tl.steer(steering_direction, magnitude=magnitude, feature_axis=-1)}


steered_tokens, steered_logs = model.generate(start, steps=3, hook_factory=hooks_for_step)
print("steered", steered_tokens)
assert clean_tokens.tolist() == [[0, 1, 1, 1]]
assert steered_tokens.tolist() == [[0, 3, 3, 3]]
assert len(clean_logs) == len(steered_logs) == 3
assert steered_logs[0].last_run_records()[-1].helper_name == "steer"